In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from torch import nn
import json
from collections import Counter

Загрузим датасет

In [2]:
tags_to_names = {
    "cs": "Computer Science",
    "econ": "Economics",
    "eess": "Electrical Engineering and Systems Science",
    "math": "Mathematics",
    "physics": "Physics",
    "q-bio": "Quantitive Biology",
    "q-fin": "Quantitive Finance",
    "stat": "Statistics"
}

In [3]:
ds_balanced = load_dataset("pinmax/arxiv_balanced_soft_labels")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/674 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/104M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/26.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/139511 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/34878 [00:00<?, ? examples/s]

Скачаем уже предобученную модель distilbert с HF

In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=len(tags_to_names))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Токенизируем abstract и title в датасете

In [5]:
def tokenize_function(examples):
    return tokenizer(examples["title"], examples["abstract"], padding="max_length", truncation=True, max_length=512)

In [6]:
ds_balanced = ds_balanced.map(tokenize_function, keep_in_memory=True)

Map:   0%|          | 0/139511 [00:00<?, ? examples/s]

Map:   0%|          | 0/34878 [00:00<?, ? examples/s]

In [7]:
ds_balanced.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

In [8]:
ds_balanced

DatasetDict({
    train: Dataset({
        features: ['title', 'abstract', 'label', 'soft_labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 139511
    })
    test: Dataset({
        features: ['title', 'abstract', 'label', 'soft_labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 34878
    })
})

In [ ]:
from huggingface_hub import login

login("")

In [ ]:
TrainingArguments?

Дообучим модель на получившемся датасете

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

args = TrainingArguments(
    output_dir="./article-classifier",
    num_train_epochs=5,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",

    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    fp16=True,

    warmup_ratio=0.1,
    label_smoothing_factor=0.1,

    logging_strategy="epoch",
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    push_to_hub=True,
    hub_model_id="pinmax/article-classifier-distilbert-big-dataset",

    disable_tqdm=False,
    dataloader_num_workers=2,

    optim="adamw_torch",
    weight_decay=0.1,
    use_cpu=False,
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_balanced["train"],
    eval_dataset=ds_balanced["test"],
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.980210,0.808851,0.843483,0.843482,0.844193,0.843483
2,0.766535,0.784106,0.855812,0.855125,0.858589,0.855812
3,0.693117,0.789651,0.857188,0.857078,0.858640,0.857188


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]